<img src="http://developer.download.nvidia.com/compute/machine-learning/frameworks/nvidia_logo.png" align="right" width="100px"/>

# Document Intelligence with Nemotron Parse + StepFun Invocation Endpoint

> **You do not need a GPU to run this notebook.** Every model call goes
> to NVIDIA's hosted chat-completions endpoint at
> `https://integrate.api.nvidia.com/v1/chat/completions`.
> All you need is a `NVIDIA_API_KEY` with access to the two model IDs
> configured in Setup below.

This notebook builds a streamlined, **all-modality** document
analysis pipeline by pairing **Nemotron Parse**
(`nvidia/nemotron-parse`) with **StepFun Flash**
(`stepfun-ai/step-3.7-flash`). Parse provides the spatial
anchoring that turns each PDF page into typed blocks and picture
bounding boxes. StepFun is the vision-language model that both
**reads each cropped picture** and **answers questions about the
whole assembled document**.

To make the case concretely, we run the pipeline against four pages
picked from **three different public PDFs** — a Pew Research report,
a social-media analytics deck, and a graduate-studies brochure —
each chosen to stress a **different content modality** that a single
page-level VLM call cannot handle alone:

| Modality | Source | Why it needs Parse |
| --- | --- | --- |
| **Chart** | Pew Research, p. 5 | a stacked-bar chart with eight policy rows — Parse isolates the bar-graph region so StepFun's transcription is a clean markdown table, not a screenshot caption |
| **Multi-picture page** | social-media report, p. 11 | three Facebook-post screenshots side by side — without Parse's bbox split, the QA call cannot tell which post is *the Disneyland post* |
| **Infographic** | social-media report, p. 20 | a dense pixel-only demographic panel — Parse draws the panel boundary, StepFun lists every number inside |
| **Structured table** | Graduate Studies brochure, p. 11 | a two-column programme table — Parse surfaces it as LaTeX-tabular text, so no vision call is needed to read it |

By the end of the tutorial you will see how the pair turns these
unstructured PDFs into **page-cited, phrase-quoted answers** that
any reader can verify against the page image.


## 1. Introduction: three roles, two model surfaces

Our pipeline uses each hosted model for its specialty around **one
unified spatial context**:

* **`nvidia/nemotron-parse` — the Architect.** A deterministic layout
  parser that returns every block's **type** (`Title`, `Text`,
  `Table`, `List-item`, `Picture`, ...), **bounding box**, and
  **reading order** in one call.

* **`stepfun-ai/step-3.7-flash` — the Visual Specialist.** Every
  `Picture` Parse identifies becomes one StepFun call that first
  *classifies* the image (`Infographic`, `Bar Graph`, `Line Graph`,
  `Smartphone Screenshot`, ...) and then *transcribes* it with a
  prompt tailored to that sub-type.

* **`stepfun-ai/step-3.7-flash` — the Reasoning Engine.** The
  same VLM reads the assembled document context: text blocks in
  reading order with picture transcriptions inlined at their spatial
  positions. It answers the question, cites the page it came from,
  and quotes the supporting phrase verbatim.

Pipeline shape:

```text
PDF page -> Nemotron Parse -> typed text/table blocks
                         -> picture boxes -> StepFun Visual Specialist
text + picture transcriptions -> reading-order page context
page contexts + question -> StepFun Reasoning Engine -> cited answer
```

### Why pair Parse with StepFun instead of calling the VLM on the whole page?

StepFun is a capable multimodal model, but real documents create four
structural problems that are easier to solve with Parse in front:

1. **Tables and headers need structure, not pixels.** On the
   Graduate-Studies brochure page, Parse emits a `Table` block with a
   LaTeX-tabular body, so the Reasoning Engine can answer directly
   from text.
2. **A chart region needs isolation before transcription.** On the
   Pew Research page, Parse draws one `Picture` bbox around the chart;
   StepFun receives only the crop and returns a clean structured
   transcription.
3. **Multi-picture pages bleed together.** Page 11 of the social-media
   report has three screenshots side by side. Parse cuts them into
   separate boxes so StepFun reads one card at a time.
4. **Citations need anchors.** Parse gives every content item a `bbox`
   and reading-order index, so answers can cite `(p. 20)` and quote the
   phrase that supports the answer.

Two design levers do the heavy lifting:

1. **Divide and conquer on every `Picture`.** One class in, many kinds
   of picture transcriptions out.
2. **Spatial-context weave.** Parse's reading-order bboxes let us
   interleave each picture's transcription at the exact position it
   occupies on the page.


## 2. Setup and prerequisites

Five Python packages are all we need: `pymupdf` for PDF rendering,
`pillow` for image handling, `requests` for the API calls, `pandas`
for tabular display, and `python-dotenv` to load the NVIDIA key from
a `.env` file.

We install with [**uv**](https://docs.astral.sh/uv/) -- the fast
package manager from Astral -- and fall back to `pip` automatically
if `uv` is not on your `PATH`.  Recommended workflow before launching
Jupyter:

```bash
# one-time install of uv (https://docs.astral.sh/uv/getting-started/installation/)
curl -LsSf https://astral.sh/uv/install.sh | sh

# create + activate an isolated environment for this notebook
uv venv .venv && source .venv/bin/activate
uv pip install jupyter
jupyter lab stepfun_doc_intelligence_with_parse.ipynb
```

The next cell installs the runtime deps into whichever environment
the notebook kernel is already pointing at, so it works whether you
ran the steps above or are using a colleague-provided kernel.


In [ ]:
import shutil, subprocess, sys
555
PKGS = ["requests", "pillow", "pymupdf", "pandas", "python-dotenv"]

if shutil.which("uv"):
    print("[setup] installing via uv ->", sys.executable)
    subprocess.check_call([
        "uv", "pip", "install", "--quiet",
        "--python", sys.executable,
        *PKGS,
    ])
else:
    print("[setup] uv not on PATH; falling back to pip. "
          "Install uv from https://docs.astral.sh/uv/ for ~10x faster syncs.")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", *PKGS,
    ])

print("[setup] OK -- runtime deps ready.")

### Configure endpoints and keys

This notebook supports a compact `.env` shape. `NVAI_CHAT_COMPLETIONS_URL` is the default NVIDIA API Catalog chat-completions endpoint for both models; set `PARSE_CHAT_COMPLETIONS_URL` or `STEPFUN_CHAT_COMPLETIONS_URL` only if you need separate endpoints.

Make the key and endpoint visible to this notebook in either of two
ways:

- **`.env` file** in this notebook's directory:
  ```bash
  NVIDIA_API_KEY=nvapi-...
  NVAI_CHAT_COMPLETIONS_URL=https://integrate.api.nvidia.com/v1/chat/completions
  STEPFUN_VLM_MODEL=stepfun-ai/step-3.7-flash
  ```
- **Shell export** before launching Jupyter:
  ```bash
  export NVIDIA_API_KEY=nvapi-...
  export NVAI_CHAT_COMPLETIONS_URL=https://integrate.api.nvidia.com/v1/chat/completions
  export STEPFUN_VLM_MODEL=stepfun-ai/step-3.7-flash
  ```

The notebook does not store credentials in source control; `.env` and
`.env.local` are ignored by this directory's `.gitignore`.


In [ ]:
from __future__ import annotations

import base64
import io
import json
import os
import re
import textwrap
import time
from pathlib import Path
from typing import Any

import fitz  # PyMuPDF
import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from PIL import Image, ImageDraw, ImageFont

NOTEBOOK_CWD = Path.cwd()
_DEMO_RELATIVE_PATH = Path("usage-cookbook") / "Nemotron-3-Nano-Omni" / "doc-intelligence-with-parse"


def _resolve_demo_root() -> Path:
    """Find this demo directory even when Jupyter starts from the repo root."""
    candidates = [
        NOTEBOOK_CWD,
        NOTEBOOK_CWD.parent if NOTEBOOK_CWD.name == "notebooks" else NOTEBOOK_CWD,
        NOTEBOOK_CWD / _DEMO_RELATIVE_PATH,
        NOTEBOOK_CWD / "Nemotron" / _DEMO_RELATIVE_PATH,
    ]
    for parent in NOTEBOOK_CWD.parents:
        candidates.extend([
            parent,
            parent / _DEMO_RELATIVE_PATH,
            parent / "Nemotron" / _DEMO_RELATIVE_PATH,
        ])

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if any((candidate / name).exists() for name in ["stepfun_doc_intelligence_with_parse.ipynb"]):
            return candidate
    return NOTEBOOK_CWD


REPO_ROOT = _resolve_demo_root()

_loaded_env_files: list[Path] = []
for _env_file in [REPO_ROOT.parent / ".env", REPO_ROOT / ".env"]:
    if _env_file.exists():
        # Load broader defaults first; let the demo-local .env win.
        load_dotenv(_env_file, override=(_env_file.parent == REPO_ROOT))
        _loaded_env_files.append(_env_file)

# Backward-compatible .env support:
#   NVIDIA_API_KEY + NVAI_CHAT_COMPLETIONS_URL are the default credential
#   and chat-completions URL for both models. Override PARSE_* or STEPFUN_*
#   only if the two calls need separate credentials or endpoints.
COMMON_API_KEY = os.environ.get("NVIDIA_API_KEY", "YOUR_API_KEY_HERE")
PARSE_API_KEY = os.environ.get("PARSE_API_KEY") or COMMON_API_KEY
STEPFUN_API_KEY = os.environ.get("STEPFUN_API_KEY") or COMMON_API_KEY

PARSE_CHAT_COMPLETIONS_URL = os.environ.get(
    "PARSE_CHAT_COMPLETIONS_URL",
    "https://integrate.api.nvidia.com/v1/chat/completions",
)
STEPFUN_CHAT_COMPLETIONS_URL = os.environ.get(
    "STEPFUN_CHAT_COMPLETIONS_URL",
    os.environ.get(
        "NVAI_CHAT_COMPLETIONS_URL",
        "https://integrate.api.nvidia.com/v1/chat/completions",
    ),
)

PARSE_MODEL = os.environ.get("PARSE_MODEL", "nvidia/nemotron-parse")
STEPFUN_VLM_MODEL = os.environ.get("STEPFUN_VLM_MODEL", "stepfun-ai/step-3.7-flash")
NVAI_REQUEST_TIMEOUT = int(os.environ.get("NVAI_REQUEST_TIMEOUT", "480"))
NVAI_MAX_RETRIES = int(os.environ.get("NVAI_MAX_RETRIES", "2"))

if not PARSE_API_KEY or PARSE_API_KEY == "YOUR_API_KEY_HERE":
    raise RuntimeError(
        "Parse API key is not set. Add NVIDIA_API_KEY=nvapi-... or "
        "PARSE_API_KEY=nvapi-... to a .env file in this notebook's directory."
    )
if not STEPFUN_API_KEY or STEPFUN_API_KEY == "YOUR_API_KEY_HERE":
    raise RuntimeError(
        "StepFun API key is not set. Add NVIDIA_API_KEY=nvapi-... or "
        "STEPFUN_API_KEY=nvapi-... to a .env file in this notebook's directory."
    )

print(f"Demo root:       {REPO_ROOT}")
print("Env files:       " + (", ".join(str(p) for p in _loaded_env_files) or "none found"))
print(f"Parse endpoint:  {PARSE_CHAT_COMPLETIONS_URL}")
print(f"StepFun endpoint:{STEPFUN_CHAT_COMPLETIONS_URL}")
print(f"Architect:       {PARSE_MODEL}")
print(f"Specialist + QA: {STEPFUN_VLM_MODEL}")
print(f"Request timeout: {NVAI_REQUEST_TIMEOUT}s, retries: {NVAI_MAX_RETRIES}")

CLASS_COLORS = {
    "Title": "#D32F2F", "Section-header": "#E91E63", "Text": "#4CAF50",
    "List-item": "#1976D2", "Caption": "#607D8B", "Table": "#03A9F4",
    "Picture": "#6D4C41", "Figure": "#6D4C41", "Formula": "#FF9800",
    "Page-header": "#9E9E9E", "Page-footer": "#9E9E9E", "Footnote": "#00BCD4",
    "Bibliography": "#512DA8", "TOC": "#FFC107", "DEFAULT": "#9E9E9E",
}


## 3. The example document set

We pick **four pages from three different public PDFs** so the
pipeline stresses a different content modality on each page, and so
the reader can visually verify every answer against the original
page image.  The exact pages are:

| `short_id` | Modality | PDF | Page |
| --- | --- | --- | --- |
| `pew`      | Chart             | `05-03-18-political-release.pdf` (Pew Research) | 5  |
| `social`   | Multi-picture     | `measuringsuccessonfacebooktwitterlinkedin-160317142140_95.pdf` (Social-Media Analytics Report) | 11 |
| `linkedin` | Infographic       | same Social-Media report as above | 20 |
| `gpl`      | Structured table  | `GPL-Graduate-Studies-Professional-Learning-Brochure-Jul-2021.pdf` (Graduate Studies brochure) | 11 |

Every page of these PDFs is a **rasterised image** — selecting text
with your mouse in a PDF viewer returns nothing, so a text-first
parser gives you nothing to work with.  This is precisely the class
of document where a VLM-driven pipeline earns its keep.

Parse's layout overlay for each of these pages is generated inline
by the pipeline in §4.2 — so the annotations you see in this
notebook are produced by the same `nvidia/nemotron-parse` call the
rest of the pipeline consumes, never a pre-baked asset.  Point
`DEMO_DOCS` at any `(pdf, page)` pairs on your disk to try your own
— the rest of the notebook does not change.

In [ ]:
DOC_DIR = REPO_ROOT / "data" / "documents"
DOC_DIR.mkdir(parents=True, exist_ok=True)

# (short_id, pdf_path, page_number, modality_label)
# short_id is a stable key used downstream to group outputs per page.
DEMO_DOCS: list[tuple[str, Path, int, str]] = [
    ("pew",      DOC_DIR / "05-03-18-political-release.pdf",                                    5,  "chart"),
    ("social",   DOC_DIR / "measuringsuccessonfacebooktwitterlinkedin-160317142140_95.pdf",     11, "multi-picture"),
    ("linkedin", DOC_DIR / "measuringsuccessonfacebooktwitterlinkedin-160317142140_95.pdf",     20, "infographic"),
    ("gpl",      DOC_DIR / "GPL-Graduate-Studies-Professional-Learning-Brochure-Jul-2021.pdf",  11, "table"),
]

# Short, human-friendly name to use in section headings (the PDF filename
# itself is too long to read in a heading).
DISPLAY_NAME = {
    "pew":      "Pew Research -- Political Release",
    "social":   "Social-Media Analytics Report",
    "linkedin": "Social-Media Analytics Report",
    "gpl":      "Graduate Studies Brochure",
}

# Public source URLs for each PDF.  The notebook is self-contained:
# if a PDF is missing, we download it once into `DOC_DIR` on first
# run.  The three demo PDFs are mirrored on the MMLongBench-Doc
# Hugging Face dataset (`yubo2333/MMLongBench-Doc`).  Point
# `DOC_DIR` at any directory you control (or pre-populate it
# yourself) and the pipeline consumes the local copy after that.
_HF_DOC_ROOT = (
    "https://huggingface.co/datasets/yubo2333/MMLongBench-Doc/"
    "resolve/main/documents"
)
_PDF_SOURCES: dict[str, str] = {
    name: f"{_HF_DOC_ROOT}/{name}" for name in {
        "05-03-18-political-release.pdf",
        "measuringsuccessonfacebooktwitterlinkedin-160317142140_95.pdf",
        "GPL-Graduate-Studies-Professional-Learning-Brochure-Jul-2021.pdf",
    }
}


def _ensure_pdf(pdf: Path) -> None:
    if pdf.exists():
        return
    url = _PDF_SOURCES.get(pdf.name)
    if url is None:
        raise FileNotFoundError(
            f"PDF not found and no default URL registered: {pdf}.  "
            "Either drop the file into DOC_DIR yourself or add a "
            "URL entry to _PDF_SOURCES.")
    print(f"  [download] {pdf.name}  <-  {url}")
    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
    except Exception as exc:
        raise FileNotFoundError(
            f"Could not auto-download {pdf.name} from {url}: {exc}.  "
            "Drop the PDF into DOC_DIR manually and re-run this cell."
        ) from exc
    pdf.write_bytes(r.content)


for sid, pdf, pn, label in DEMO_DOCS:
    _ensure_pdf(pdf)
    print(f"  [{sid:8s}] p.{pn:<3d} {label:<14s} -> {pdf.name}  "
          f"({pdf.stat().st_size / 1024:,.0f} KB)")

## 4. The core pipeline in action

Now, let's walk through the code that powers the pipeline.

### 4.1. Helper functions

One self-contained cell with every building block the pipeline needs,
organised into three groups:

1. **Imaging** — `pdf_page_to_image` renders a page to pixels,
   `pil_to_data_url` encodes it for the API, `draw_annotations` paints
   bounding-box overlays (with per-class colours and sub-typed
   picture labels) on top of any page.
2. **Model surfaces** — `call_nemotron_parse` for the Architect,
   `call_stepfun_vlm` as a single entry point that serves both the
   Visual Specialist and the Reasoning Engine over the NVIDIA hosted
   chat-completions endpoint, plus small helpers to
   pull clean text or JSON out of the response.
3. **Pipeline stages** — `describe_picture` implements the
   divide-and-conquer lever (classify, then dispatch to a
   content-aware prompt); `assemble_page_context` implements the
   spatial weave (interleaves picture transcriptions into the page's
   prose at their reading-order position); `ask_question` is the
   final Reasoning Engine call, with three short answer rules that
   make every answer page-cited and phrase-quoted.


In [ ]:
# =============================================================================
# Group 1 — Imaging
# =============================================================================

def pdf_page_to_image(pdf_path: str | Path, page_index: int, *, dpi: int = 150) -> Image.Image:
    """Render a 0-indexed PDF page to an RGB PIL image at `dpi`."""
    doc = fitz.open(pdf_path)
    try:
        page = doc.load_page(page_index)
        zoom = dpi / 72.0
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
        return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    finally:
        doc.close()


def pil_to_data_url(img: Image.Image, *, fmt: str = "JPEG", quality: int = 85) -> str:
    """JPEG- or PNG-encode a PIL image and wrap it as a `data:` URL."""
    if img.mode != "RGB":
        img = img.convert("RGB")
    buf = io.BytesIO()
    if fmt.upper() == "JPEG":
        img.save(buf, format="JPEG", quality=quality)
        mime = "image/jpeg"
    else:
        img.save(buf, format="PNG")
        mime = "image/png"
    return f"data:{mime};base64," + base64.b64encode(buf.getvalue()).decode()


def _luminance(hex_color: str) -> float:
    h = hex_color.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return 0.2126 * r + 0.7152 * g + 0.0722 * b


def draw_annotations(image: Image.Image, blocks: list[dict[str, Any]]) -> Image.Image:
    """Paint labelled bounding boxes for every block.  Pictures that
    carry a `sub_type` (from the Visual Specialist) are labelled as
    `Picture:<sub_type>` rather than just `Picture`.
    """
    out = image.copy()
    draw = ImageDraw.Draw(out)
    W, H = out.size
    box_w = max(2, int(W / 600))
    font_size = max(14, int(W / 80))
    try:
        font = ImageFont.truetype("Arial.ttf", font_size)
    except Exception:
        font = ImageFont.load_default()
    for i, b in enumerate(blocks):
        bb = b.get("bbox") or {}
        x0, y0 = bb.get("xmin", 0.0) * W, bb.get("ymin", 0.0) * H
        x1, y1 = bb.get("xmax", 0.0) * W, bb.get("ymax", 0.0) * H
        if x1 <= x0 or y1 <= y0:
            continue
        cat = b.get("type", "DEFAULT")
        sub = b.get("sub_type")
        color = CLASS_COLORS.get(cat, CLASS_COLORS["DEFAULT"])
        label = f"{i}:{cat}" + (f":{sub}" if sub else "")
        draw.rectangle([x0, y0, x1, y1], outline=color, width=box_w)
        try:
            tb = draw.textbbox((0, 0), label, font=font)
            tw, th = tb[2] - tb[0], tb[3] - tb[1]
        except Exception:
            tw, th = len(label) * 7, font_size
        bg = (x0, max(0, y0 - th - 6), x0 + tw + 10, max(0, y0 - 6))
        draw.rectangle(bg, fill=color)
        text_color = "#000000" if _luminance(color) > 140 else "#FFFFFF"
        draw.text((bg[0] + 5, bg[1] + 2), label, fill=text_color, font=font)
    return out


# =============================================================================
# Group 2 — Model surfaces
# =============================================================================

def _headers(api_key: str) -> dict[str, str]:
    return {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }


def _post_chat_completion(
    body: dict[str, Any],
    *,
    endpoint: str,
    api_key: str,
    timeout: int | None = None,
) -> requests.Response:
    """POST to a chat-completions endpoint with small retry protection."""
    timeout = timeout or NVAI_REQUEST_TIMEOUT
    for attempt in range(NVAI_MAX_RETRIES + 1):
        try:
            return requests.post(
                endpoint,
                headers=_headers(api_key),
                json=body,
                timeout=timeout,
            )
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            if attempt >= NVAI_MAX_RETRIES:
                raise
            sleep_s = 2 ** attempt
            print(f"  [retry] hosted endpoint timed out; retrying in {sleep_s}s")
            time.sleep(sleep_s)


def call_nemotron_parse(image: Image.Image) -> list[dict[str, Any]]:
    """Run `nemotron-parse` on a page image.  Returns a flat list of
    blocks with `type`, `bbox`, and `text`.
    """
    body = {
        "model": PARSE_MODEL,
        "messages": [{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": pil_to_data_url(image, fmt="PNG")}}
        ]}],
        "tools": [{"type": "function", "function": {"name": "markdown_bbox"}}],
        "tool_choice": {"type": "function", "function": {"name": "markdown_bbox"}},
        "max_tokens": 8192,
        "temperature": 0.0,
    }
    r = _post_chat_completion(
        body,
        endpoint=PARSE_CHAT_COMPLETIONS_URL,
        api_key=PARSE_API_KEY,
    )
    r.raise_for_status()
    args = r.json()["choices"][0]["message"]["tool_calls"][0]["function"]["arguments"]
    parsed = json.loads(args)
    blocks = parsed if isinstance(parsed, list) else parsed.get("tool_call_arguments", [])
    if blocks and isinstance(blocks[0], list):
        blocks = blocks[0]
    return blocks or []


# StepFun can answer directly, but VLMs sometimes include reasoning-style
# preambles when asked to transcribe dense visual content.  The Specialist
# wants the final transcription only, so direct calls get a small system
# guard.  `extract_text` also strips any <think> blocks or echoed guard text.
_SYS_NO_THINK = (
    "/no_think\n"
    "Answer directly and concisely. Do NOT include any reasoning, "
    "preamble, or <think> blocks."
)
_SYSTEM_ECHO = re.compile(
    r"^\s*(?:/no_think\s*)?Answer directly and concisely[^.]*\.\s*"
    r"Do NOT include any reasoning[^.]*\.\s*",
    re.IGNORECASE,
)
_LEAK_HEADS = (
    "okay,", "okay ", "the user wants", "let me ", "first,",
    "i need to", "i'll ", "i will ", "alright,",
)


def call_stepfun_vlm(
    prompt: str,
    images: list[Image.Image] | None = None,
    *,
    direct: bool = True,
    json_mode: bool = False,
    temperature: float = 0.2,
    top_p: float = 0.95,
    max_tokens: int = 2048,
) -> dict[str, Any]:
    """One StepFun VLM call, used by both the Specialist and the
    Reasoning Engine.  `direct=True` adds a short system guard that
    asks for final answers only.  The request uses standard
    chat-completions fields so it can target
    `stepfun-ai/step-3.7-flash` through NVIDIA's hosted endpoint.
    """
    parts: list[dict[str, Any]] = [{"type": "text", "text": prompt}]
    for img in images or []:
        parts.append({"type": "image_url", "image_url": {"url": pil_to_data_url(img)}})
    messages: list[dict[str, Any]] = []
    if direct:
        messages.append({"role": "system", "content": _SYS_NO_THINK})
    messages.append({"role": "user", "content": parts})
    body: dict[str, Any] = {
        "model": STEPFUN_VLM_MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": top_p,
        "stream": False,
    }
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    r = _post_chat_completion(
        body,
        endpoint=STEPFUN_CHAT_COMPLETIONS_URL,
        api_key=STEPFUN_API_KEY,
    )
    if r.status_code >= 400 and json_mode and "response_format" in body:
        # Some hosted VLMs ignore JSON prompts but reject response_format.
        # Retry once with the prompt-only JSON instruction.
        body.pop("response_format", None)
        r = _post_chat_completion(
            body,
            endpoint=STEPFUN_CHAT_COMPLETIONS_URL,
            api_key=STEPFUN_API_KEY,
        )
    r.raise_for_status()
    return r.json()


_THINK_RE = re.compile(r"<think\b[^>]*>.*?</think>", re.DOTALL | re.IGNORECASE)


def extract_text(resp: dict[str, Any]) -> str:
    msg = resp.get("choices", [{}])[0].get("message", {})
    text = (
        msg.get("content")
        or msg.get("reasoning")
        or msg.get("reasoning_content")
        or ""
    ).strip()
    text = _THINK_RE.sub("", text).strip()
    text = _SYSTEM_ECHO.sub("", text, count=1).strip()
    return text


def _is_leaky(text: str) -> bool:
    """Heuristic: the Specialist sometimes leaks chain-of-thought
    into `content` without `<think>` tags.  If the response opens
    with a classic reasoning preamble, we retry the call once.
    """
    head = text.lstrip().lower()[:80]
    return any(head.startswith(p) for p in _LEAK_HEADS)


def extract_json_object(resp: dict[str, Any]) -> dict[str, Any]:
    """Extract a JSON object robustly.  Looks in both `content` and
    `reasoning` fields (the hosted endpoint occasionally collapses
    JSON-mode output into the reasoning stream), strips code fences,
    and falls back to the outermost `{...}` substring.
    """
    msg = resp.get("choices", [{}])[0].get("message", {})
    for raw in (
        msg.get("content") or "",
        msg.get("reasoning") or "",
        msg.get("reasoning_content") or "",
    ):
        s = _THINK_RE.sub("", (raw or "").strip()).strip()
        if not s:
            continue
        if s.startswith("```"):
            s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE).strip("` \n")
        try:
            obj = json.loads(s)
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            pass
        l, r = s.find("{"), s.rfind("}")
        if l != -1 and r > l:
            try:
                obj = json.loads(s[l : r + 1])
                if isinstance(obj, dict):
                    return obj
            except json.JSONDecodeError:
                pass
    return {}


# =============================================================================
# Group 3 — Pipeline stages
# =============================================================================

CLASSIFY_PROMPT = (
    "Analyze the provided image and classify its content. Your response "
    "MUST be a single, valid JSON object with the following keys:\n"
    '- "image_type": one of "Extractive" (charts, graphs, diagrams, '
    'tables, flowcharts, maps) or "Descriptive" (photographs, '
    "illustrations, artistic pieces).\n"
    '- "sub_type": a specific label, e.g. "Line Graph", "Bar Graph", '
    '"Infographic", "Flowchart", "Pyramid Diagram", "Smartphone '
    'Screenshot", "Photograph".\n'
    '- "subject_matter": one-sentence summary of the picture topic.\n'
    '- "contains_text": boolean, true if the image has readable text.\n'
    "Provide ONLY the JSON object and nothing else."
)

# Divide-and-conquer dispatch table — different picture kinds call for
# different transcription prompts.
ANALYSIS_PROMPTS: dict[tuple[str, str], str] = {
    ("Extractive", "Default"): (
        "Analyze the provided image and extract all structured "
        "information.  If the information fits a tabular format, "
        "render it as a Markdown table.  Otherwise, produce a concise "
        "summary capturing every number, label, and relationship."
    ),
    ("Extractive", "Line Graph"): (
        "You are a data analyst.  Transcribe the data from this line "
        "graph.  State the title, X- and Y-axis labels, and for each "
        "series extract the data points as [x, y] pairs.  Return one "
        "JSON object."
    ),
    ("Extractive", "Bar Graph"): (
        "You are a data analyst.  Transcribe this bar chart.  State "
        "the title, axis labels, category names, and the value for "
        "each bar.  Return both a Markdown table AND a one-sentence "
        "headline finding."
    ),
    ("Extractive", "Infographic"): (
        "Transcribe this infographic.  For each panel or section, "
        "list its name and every labelled value or percentage.  "
        "Preserve the grouping the designer used.  Return a "
        "structured Markdown outline."
    ),
    ("Extractive", "Flowchart"): (
        "Transcribe this flowchart or diagram.  List every node and "
        "every labelled edge.  State the overall flow direction."
    ),
    ("Extractive", "Pyramid Diagram"): (
        "Transcribe this pyramid diagram.  List every tier from top "
        "to bottom with its label and any supporting text.  Infer "
        "the ordering or progression the diagram communicates."
    ),
    ("Descriptive", "Default"): (
        "Describe this image in detail: subject matter, composition, "
        "colours, and any text visible in the scene.  Do not "
        "speculate beyond what is visible."
    ),
    ("Descriptive", "Smartphone Screenshot"): (
        "Transcribe this smartphone UI.  Read every label, button, "
        "menu entry, and message visible.  Infer the app or screen "
        "type (e.g. SMS, contacts, home screen, call UI) and list "
        "the UI elements in reading order."
    ),
}


def pick_analysis_prompt(classification: dict[str, Any]) -> str:
    it = classification.get("image_type", "Extractive")
    st = classification.get("sub_type", "Default")
    return (ANALYSIS_PROMPTS.get((it, st))
            or ANALYSIS_PROMPTS.get((it, "Default"))
            or ANALYSIS_PROMPTS[("Extractive", "Default")])


def crop_picture(page: Image.Image, bbox: dict[str, float]) -> Image.Image:
    W, H = page.size
    return page.crop((int(bbox["xmin"] * W), int(bbox["ymin"] * H),
                      int(bbox["xmax"] * W), int(bbox["ymax"] * H)))


def describe_picture(crop: Image.Image) -> dict[str, Any]:
    """Divide-and-conquer: classify first, then transcribe with a
    content-aware prompt.  Two API calls per picture.  If the first
    transcription leaks reasoning into `content`, we retry once --
    this small resilience step keeps the final context clean so the
    Reasoning Engine doesn't echo chain-of-thought back in §7.
    """
    cls_resp = call_stepfun_vlm(
        CLASSIFY_PROMPT, images=[crop],
        direct=True, json_mode=True,
        temperature=0.0, max_tokens=1024,
    )
    classification = extract_json_object(cls_resp) or {
        "image_type": "Extractive", "sub_type": "Default",
    }
    prompt = pick_analysis_prompt(classification)
    desc_resp = call_stepfun_vlm(
        prompt, images=[crop],
        direct=True, json_mode=False,
        temperature=0.2, max_tokens=2048,
    )
    description = extract_text(desc_resp)
    if _is_leaky(description):
        desc_resp = call_stepfun_vlm(
            prompt, images=[crop],
            direct=True, json_mode=False,
            temperature=0.2, max_tokens=2048,
        )
        description = extract_text(desc_resp)
    return {"classification": classification, "description": description}


def assemble_page_context(
    parse_blocks: list[dict[str, Any]],
    picture_descriptions: list[str],
    *,
    page_n: int,
) -> str:
    """Spatial weave: walk parse's blocks in reading order, emit each
    text block verbatim and substitute each `Picture` block with its
    StepFun VLM transcription inline at the same spatial position.

    Pages are wrapped in strong visual delimiters so the Reasoning
    Engine never confuses which page a fact came from when it is asked
    to cite it.
    """
    header = f"===== PAGE {page_n} ====="
    footer = f"===== END PAGE {page_n} ====="
    lines = [header]
    pic_idx = 0
    for b in parse_blocks:
        t = b.get("type")
        if t == "Picture":
            if pic_idx < len(picture_descriptions):
                lines.append(f"[Picture on page {page_n}] "
                             + picture_descriptions[pic_idx])
                pic_idx += 1
        elif b.get("text"):
            lines.append(b["text"])
    lines.append(footer)
    return "\n\n".join(lines)


QA_PROMPT_TEMPLATE = (
    "Based on the following document context, please answer the "
    "question that follows.\n\n"
    "- DOCUMENT CONTEXT\n{context}\n\n"
    "- QUESTION\n{question}\n\n"
    "Answer rules:\n"
    "  1. Cite the page number(s) the answer comes from in the form "
    "'(p. <page-number>)' immediately after the answer.\n"
    "  2. Quote the specific phrase or value from the document "
    "context that supports the answer.\n"
    "  3. If the answer is NOT present in the context, output exactly "
    "the string `Not answerable` — do not guess."
)


def ask_question(question: str, context: str) -> str:
    """Run the Reasoning Engine.  We keep `direct=True` + the
    `/no_think` system message so the final answer is a clean
    citation, not a transcript of the model's inner monologue.  The
    Specialist's transcriptions already carried the heavy visual
    extraction at pipeline time, so the QA call only needs to search
    the assembled context for the cited phrase.
    """
    prompt = QA_PROMPT_TEMPLATE.format(context=context, question=question)
    resp = call_stepfun_vlm(
        prompt, images=None,
        direct=True, json_mode=False,
        temperature=0.2, max_tokens=1024,
    )
    answer = extract_text(resp)
    if _is_leaky(answer):
        resp = call_stepfun_vlm(
            prompt, images=None,
            direct=True, json_mode=False,
            temperature=0.2, max_tokens=1024,
        )
        answer = extract_text(resp)
    return answer


print("Helpers ready.")


### 4.2. Executing the pipeline

Now we drive the four pages through the pipeline.  The loop below:

1. Renders each PDF page to pixels.
2. Makes one **Stage 1** call per page to the Architect
   (`nemotron-parse`) and **displays the page with Parse's coloured
   layout boxes overlaid**.  That overlay *is* our proof of spatial
   anchoring: you can see Parse split the social-media page into
   three separate `Picture` boxes, wrap the Pew chart in a single
   `Picture:Bar Graph` box, and flip the GPL table into a
   `Table` block with no `Picture` call at all.
3. For every `Picture` block the Architect returns, crops the
   picture and makes a pair of **Stage 2** calls to the Visual
   Specialist — one to classify the picture's sub-type, one to
   transcribe it with a sub-type-aware prompt.  (The crop-plus-
   transcription receipts are shown modality-by-modality in §5, to
   avoid re-displaying the same images twice.)
4. Aggregates everything into a single `file_results` JSON structure
   we can inspect and query later.

Watch especially page 11 of the social-media report: the Architect
returns **three** `Picture` boxes — one per Facebook-post
screenshot — and each gets its own targeted transcription.  That is
the multi-picture isolation that makes the *"Disneyland"* question
in §7 answerable with a precise number.

In [ ]:
# Per-(short_id, page_n) page images and Parse blocks; per-pdf result bundles
page_images: dict[str, Image.Image] = {}
page_blocks: dict[str, list[dict[str, Any]]] = {}
file_results: dict[str, dict[str, Any]] = {}

# Annotated-preview width (kept small so the notebook file stays lean).
_ANNOT_W = 820

for sid, pdf, page_n, modality in DEMO_DOCS:
    print("\n" + "=" * 74)
    print(f"[{sid}] {pdf.name} -- page {page_n}  ({modality})")
    print("=" * 74)

    page = pdf_page_to_image(pdf, page_n - 1, dpi=150)
    page_images[sid] = page

    # Stage 1 -- Architect.
    t0 = time.time()
    blocks = call_nemotron_parse(page)
    type_counts: dict[str, int] = {}
    for b in blocks:
        type_counts[b.get("type", "?")] = type_counts.get(b.get("type", "?"), 0) + 1
    print(f"[Architect]  {len(blocks)} blocks in {time.time()-t0:.1f}s  "
          f"types -> {type_counts}")
    page_blocks[sid] = blocks

    # Show the coloured layout overlay in-place -- this is the annotated
    # view §3 promised.  Annotating a copy keeps sub_type labels out
    # of the overlay until the Specialist runs below.
    annotated = draw_annotations(page, blocks)
    display(Markdown(
        f"**Parse overlay** &mdash; {DISPLAY_NAME[sid]} (p. {page_n}, "
        f"modality: {modality})"))
    display(annotated.resize(
        (_ANNOT_W, int(_ANNOT_W * annotated.height / annotated.width))))

    bundle = file_results.setdefault(
        pdf.name, {"source_filename": pdf.name, "pages": []})
    page_entry: dict[str, Any] = {
        "page_number": page_n, "status": "Layout extraction successful",
        "content": [],
    }

    # Stage 2 -- Visual Specialist for every Picture block.
    n_pics = sum(1 for b in blocks if b.get("type") == "Picture")
    if n_pics == 0:
        print("[Specialist] skipped -- no Picture blocks on this page "
              "(Parse handled this modality as structured text).")

    for i, b in enumerate(blocks):
        item: dict[str, Any] = {
            "extraction_id": i, "type": b.get("type"),
            "bbox": b.get("bbox"), "text": b.get("text"),
        }
        if b.get("type") == "Picture" and b.get("bbox"):
            crop = crop_picture(page, b["bbox"])
            t1 = time.time()
            result = describe_picture(crop)
            sub = result["classification"].get("sub_type", "?")
            print(f"[Specialist] Picture #{i:<2d}  classified as '{sub}'  "
                  f"-> described in {time.time()-t1:.1f}s  "
                  f"({len(result['description'])} chars)")
            b["sub_type"] = sub
            item["classification"] = result["classification"]
            item["description"] = result["description"]
        page_entry["content"].append(item)

    bundle["pages"].append(page_entry)

_n_pics_total = sum(1 for b in file_results.values()
                    for p in b["pages"] for it in p["content"]
                    if it.get("description"))
print("\n" + "=" * 74)
print(f"Pipeline finished: {len(DEMO_DOCS)} pages across "
      f"{len(file_results)} documents, {_n_pics_total} pictures transcribed.")

## 5. Divide and conquer, modality by modality

§4.2 showed the *coloured overlays* — that is, **where** Parse drew
its boxes.  This section shows **what the pair actually produced**
inside each box, one modality at a time.  For every demo page we
pull out:

1. the **exact image crop** Parse handed to the Visual Specialist
   (or the text block if no vision call was needed), and
2. the **Specialist's transcription** of that crop — the text the
   Reasoning Engine will read in §7.

That crop-plus-transcription pair is the *receipt* of the divide-
and-conquer design.  Four modalities, four receipts:

* **Chart** – one `Picture:Bar Graph` box → a clean Markdown table.
* **Multi-picture page** – *three* `Picture` boxes → three separate
  Facebook-post transcriptions (this is the one to linger on — with
  Parse turned off, the single page-level call has to juggle three
  cards at once).
* **Infographic** – one `Picture:Infographic` box → a structured
  panel-by-panel breakdown of the LinkedIn demographic stats.
* **Structured table** – **no `Picture` call at all**: Parse emits
  the programme list directly as a LaTeX-tabular `Table` block the
  Reasoning Engine reads verbatim.

Because the Specialist's transcription is the *same text* the
Reasoning Engine reads in §7, any answer it returns can be
back-traced to one of these crops with one glance.

In [ ]:
# Per-modality evidence: the crop(s) Parse isolated + the Visual
# Specialist's transcription.  We do NOT re-display full annotated
# pages here -- those live in section 3.  The page image itself is
# shown only to provide the crop; transcriptions are rendered as
# Markdown so tables come out formatted.
MAX_DESC_CHARS = 1400

for sid, pdf, page_n, modality in DEMO_DOCS:
    display(Markdown(
        f"### {modality.title()} &mdash; {DISPLAY_NAME[sid]} (p. {page_n})"
    ))

    page = page_images[sid]
    blocks = page_blocks[sid]
    page_entry = next(p for p in file_results[pdf.name]["pages"]
                      if p["page_number"] == page_n)

    pic_items = [it for it in page_entry["content"]
                 if it.get("type") == "Picture" and it.get("description")]
    tab_items = [it for it in page_entry["content"]
                 if it.get("type") == "Table" and it.get("text")]

    # Structured-table case: Parse handled it directly; no vision
    # call was required.
    if not pic_items and tab_items:
        display(Markdown(
            "Parse surfaced this region as a `Table` block with "
            "LaTeX-tabular body &mdash; **no Visual Specialist call "
            "was needed**.  The Reasoning Engine reads the cells "
            "below as plain text:"))
        print(tab_items[0]["text"])
        continue

    block_by_id = {i: b for i, b in enumerate(blocks)}
    crop_w = 360 if len(pic_items) > 1 else 620

    for item in pic_items:
        block = block_by_id[item["extraction_id"]]
        cls = item["classification"] or {}
        sub = cls.get("sub_type", "?")
        subject = cls.get("subject_matter", "")
        header = (f"**Crop #{item['extraction_id']}** &mdash; "
                  f"classified as `Picture:{sub}`")
        if subject:
            header += f"  \n*{subject}*"
        display(Markdown(header))

        crop = crop_picture(page, block["bbox"])
        display(crop.resize((crop_w,
                             int(crop_w * crop.height / crop.width))))

        desc = item["description"]
        if len(desc) > MAX_DESC_CHARS:
            desc = desc[:MAX_DESC_CHARS].rstrip() + "\n\n*... (truncated)*"
        display(Markdown(desc))

## 6. Examining the final JSON output

The pipeline fuses both models' results into a single, structured
JSON object *per document*.  Every page carries a list of `content`
items; `Picture` items contain the Visual Specialist's
`classification` (image type, sub-type, subject-matter summary)
alongside its textual `description`, while `Table` items carry the
LaTeX-tabular text Parse emitted directly.

This JSON is the *only* artefact the Reasoning Engine needs — it
can be serialised, cached, re-queried, or piped into downstream
retrieval systems.  Below we preview the Pew Research document's
output and save one `<stem>.parse_stepfun.json` per PDF to disk.


In [ ]:
# Preview the shortest bundle (the Pew one-pager) so the reader
# sees the full shape on screen without scrolling.
_preview_pdf_name = DEMO_DOCS[0][1].name  # Pew Research release
_preview = file_results[_preview_pdf_name]
print(f"=== preview: {_preview_pdf_name} ===")
for line in json.dumps(_preview, indent=2, default=str).split("\n")[:60]:
    print(line)
print("... (truncated)\n")

# Save one JSON per PDF.  Downstream retrieval systems can index these
# directly, one embedding per page.
OUTPUT_DIR = Path("output_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for pdf_name, bundle in file_results.items():
    out = OUTPUT_DIR / (Path(pdf_name).stem + ".parse_stepfun.json")
    out.write_text(json.dumps(bundle, indent=2, default=str))
    print(f"  saved  {out.name}  ({len(bundle['pages'])} page(s))")

## 7. Querying the document with the Reasoning Engine

With every page distilled into a picture-transcribed,
spatially-ordered context, we can ask real questions.  The *same*
StepFun VLM we used as Visual Specialist now puts on its
**Reasoning Engine** hat and answers — citing the page and quoting
the supporting phrase for every answer.

We pose one question per modality; each answer is visible on the
annotated page tile in §3, so you can verify it by eye:

1. **Chart (Pew Research, p. 5).**  Which policy area draws the
   largest share of *very* confident respondents?  A stacked-bar
   chart question that only works because the Specialist turned the
   chart into a Markdown table first.
2. **Multi-picture (Social-Media report, p. 11).**  *"How many
   likes does the Disneyland post have?"*  Three Facebook-post
   screenshots sit side by side on the page; with Parse turned off,
   a single page-level VLM call would have to pick the right card
   while reading it.  Parse's three separate `Picture` boxes make
   the answer surgical.
3. **Infographic (Social-Media report, p. 20).**  *"What percentage
   of LinkedIn users have household income above $75K?"*  The digit
   lives inside one panel of a pixel-only demographic infographic.
4. **Structured table (Graduate Studies brochure, p. 11).**  *"Which
   leadership programme has the longest Full-Time duration?"*  Pure
   Architect work — Parse preserved the programme/duration table as
   LaTeX-tabular text so the Reasoning Engine can scan it without a
   vision call.


In [ ]:
# Per-document context: each PDF we processed gets its own assembled
# context string, stitched together from the pages we ran through
# the pipeline, with strong page delimiters so citations land on the
# right page number.
doc_contexts: dict[str, str] = {}
for sid, pdf, page_n, _mod in DEMO_DOCS:
    entry = next(p for p in file_results[pdf.name]["pages"]
                 if p["page_number"] == page_n)
    descs = [it["description"] for it in entry["content"]
             if it.get("type") == "Picture" and it.get("description")]
    chunk = assemble_page_context(page_blocks[sid], descs, page_n=page_n)
    doc_contexts.setdefault(pdf.name, []).append(chunk)

for pdf_name, chunks in doc_contexts.items():
    merged = "\n\n".join(chunks)
    doc_contexts[pdf_name] = merged
    print(f"context: {pdf_name}  -> {len(merged):,} chars, "
          f"{len(chunks)} page(s)")
print()

# One question per demo page, routed to the doc whose context can
# answer it.  (short_id, question)
qa_plan: list[tuple[str, str]] = [
    ("pew",
     "Which policy area has the largest share of respondents who are "
     "'very' confident in Donald Trump? Give the policy phrase and the "
     "percentage."),
    ("social",
     "How many people like the Disneyland post?"),
    ("linkedin",
     "What percentage of LinkedIn users have a household income above "
     "$75K?"),
    ("gpl",
     "Among the leadership programmes listed, which programme has the "
     "longest Full-Time duration, and what is that duration?"),
]

# Each question fires a single Reasoning Engine call scoped to the
# document it belongs to, so citations point at the right page.
sid_to_pdf = {sid: pdf.name for sid, pdf, _, _ in DEMO_DOCS}
for sid, q in qa_plan:
    pdf_name = sid_to_pdf[sid]
    context = doc_contexts[pdf_name]
    print("=" * 74)
    print(f"DOC:      {DISPLAY_NAME[sid]}  ({pdf_name})")
    print(f"QUESTION: {q}")
    t0 = time.time()
    answer = ask_question(q, context)
    print(f"[{time.time()-t0:.1f}s]")
    display(Markdown(f"**ANSWER:**\n\n{answer}"))
    print()

## 8. Next steps

This pipeline fits on one page and covers every modality a real
PDF throws at you (text, tables, charts, infographics, UI
screenshots) — on a single StepFun VLM surface — because
the assembled context can be queried in one hosted VLM call for
the small demo set, then extended with retrieval for larger document collections.

A few directions to take it from here:

- **Try it on your documents.**  Extend `DEMO_DOCS` in §3 with any
  `(short_id, pdf, page, modality)` tuples you like and re-run.
  Pages that are rasterised images (scanned reports, marketing
  decks, social-media screenshots, dashboards) are the pipeline's
  sweet spot — they are also exactly the pages most text-first
  parsers drop on the floor.
- **Batch the picture stage when quota is tight.**  For
  picture-dense documents (dashboards, slide decks, screenshot
  catalogues), you can send every picture on a page in a single
  multi-image StepFun VLM call and ask for one JSON entry per
  picture.  This trades a little peak quality for roughly 4×
  fewer picture-stage API calls on picture-dense pages; use it
  when API-quota economics matter more than peak accuracy.
- **Scale past ~25 pages with long-context retrieval.**  The
  per-document JSON written in §6 is a ready-made input for a
  vector store: one embedding per page, retrieve the top-k at
  question time, and feed those pages' `assemble_page_context`
  strings straight into the Reasoning Engine.  Parse's reading
  order + bbox anchors survive retrieval, so the model can still
  cite the right page.
- **Customise the picture-stage prompts.**  The `ANALYSIS_PROMPTS`
  dispatch table in §4.1 is the single place where you teach the
  pipeline what a *good* transcription looks like for your domain
  — product photos, engineering drawings, medical charts, CAD
  screenshots.  Adding a new entry is three lines of code.

Two hosted model surfaces.  Three roles.  StepFun handles both
"what does this picture mean" and "what is the user actually
asking" — that is the all-modality foundation this cookbook demonstrates.
